# IDAI-780 Capstone — Checkpoint 3: Order-Controlled Search Agent
**Ryan Khan** · *Chain-of-Thought Reasoning Bias and Search-Agent Retrieval in LLM Financial Fraud Detection*

Runs the CP3 pipeline end to end: tests → classical training snapshot → Qwen3-8B grid (Direct, CoT perf-first / quality-first × none / static RAG / search agent) → Claude oracle → evaluation and figures.

**Setup:** Runtime → Change runtime type → **A100** (or L4). Add the secret `ANTHROPIC_API_KEY` (🔑, turn on notebook access). Set `REPO_URL` below, then run the cells in order. Results are written into the repo folder on your Drive, so a disconnect does not lose them.

## 0. Mount Drive and get the repo

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
import os, sys, subprocess
REPO_URL = "https://github.com/<your-username>/fraud-cot-search-agent.git"   # <- set this
PROJECT_DIR = "/content/drive/MyDrive/fraud-cot-search-agent"
if not os.path.exists(os.path.join(PROJECT_DIR, "src", "run_agent.py")):
    subprocess.run(["git", "clone", REPO_URL, PROJECT_DIR], check=True)
os.chdir(PROJECT_DIR); sys.path.insert(0, PROJECT_DIR)
!git pull -q || true
!ls

## 1. Environment

In [ ]:
!pip install -q -r requirements.txt
!nvidia-smi --query-gpu=name,memory.total --format=csv
import torch, transformers, xgboost, sklearn, anthropic
print(f"Python {sys.version.split()[0]} | torch {torch.__version__} | transformers {transformers.__version__} | "
      f"xgboost {xgboost.__version__} | scikit-learn {sklearn.__version__} | anthropic {anthropic.__version__}")
gpu_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
os.environ["HF_4BIT"] = "1" if gpu_gb < 30 else "0"
print(f"GPU {gpu_gb:.0f} GB -> 4-bit = {os.environ['HF_4BIT']}")

## 2. Unit tests (parser, budget cutoff, leakage guards, retrieval determinism, schema)

In [ ]:
!python -m pytest -q

## 3. Training snapshot — classical baselines
Model selection on 1991–97 → 2000–01, refit on 1991–2001, test once on 2003–08. Per-round loss and validation curves, XGBoost checkpoints every 50 rounds, and the full log are saved under `results/`.

In [ ]:
!python -m src.train_classical

In [ ]:
from IPython.display import Image, display
display(Image("results/figures/training_curves.png"))
!ls -la results/checkpoints/B2_xgboost_raw28
!head -5 results/training_curves/B2_xgboost_fit_vs_val.csv
!tail -12 results/logs/train_classical.log

## 4. Qwen3-8B — dev split (fits the red-flag threshold τ only; 2000–01 cases, never test)

In [ ]:
OPEN_MODEL = "Qwen/Qwen3-8B"
!python -m src.run_agent --backend hf --model {OPEN_MODEL} --split dev --n 20 --batch-size 16

## 5. Qwen3-8B — full grid on the out-of-sample test cases
7 conditions × N balanced cases. About 45–60 min on an A100 for N = 60.

In [ ]:
N_TEST = 60
!python -m src.run_agent --backend hf --model {OPEN_MODEL} --split test --n {N_TEST} --batch-size 16

## 6. Demo batch for the video (4 cases, both orders, search agent, live trace)

In [ ]:
!python -m src.run_agent --backend hf --model {OPEN_MODEL} --n 4 --conditions perf-agent qual-agent --verbose --run-tag demo
!python -m src.show_trace results/agent_hf_Qwen-Qwen3-8B_test_demo.jsonl --condition qual-agent --max 1

## 7. Claude oracle (native tool use; optional — check the API budget)

In [ ]:
from google.colab import userdata
os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")
CLAUDE_MODEL = "claude-sonnet-5-5"
!python -m src.run_agent --backend claude --model {CLAUDE_MODEL} --split dev --n 20
!python -m src.run_agent --backend claude --model {CLAUDE_MODEL} --split test --n {N_TEST}

## 8. Evaluation: baselines vs. search agent, order bias, knowledge gain

In [ ]:
!python -m src.evaluate_agent --tau-from-dev --boot 1000

In [ ]:
display(Image("results/figures/comparison.png"))
display(Image("results/figures/order_delta.png"))
from IPython.display import Markdown
display(Markdown(open("results/tables/comparison.md").read()))

## 9. Save and push results
Commit the results tables/figures so the README can show them (`git push` needs a GitHub token or do it from your computer).

In [ ]:
!git add results/tables results/figures results/training_curves results/logs results/*.json results/*.csv
!git status --short | head -30